In [8]:
# ============================================================
# 1. IMPORTS AND SETUP
# ============================================================

import sys
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg")

print("Project root exists:", project_root.exists())
print("Src exists:", (project_root / "src").exists())
print("Data pipeline exists:", (project_root / "src" / "data_pipeline.py").exists())
print("Database exists:", (project_root / "NordPoool" / "data" / "thesis_database.db").exists())

sys.path.append(str(project_root))

from src.data_pipeline import DatabaseManager, DataLoader, DataPreprocessor

db_path = project_root / "NordPoool" / "data" / "thesis_database.db"

db = DatabaseManager(db_path)
loader = DataLoader(db)
pre = DataPreprocessor()

print("Setup completed")

Project root exists: True
Src exists: True
Data pipeline exists: True
Database exists: True
Setup completed


In [12]:
# ============================================================
# 2. LOAD DATA
# ============================================================

df_prices = loader.load_prices()
df_volumes = loader.load_volumes()
df_flows = loader.load_flows()
df_capacities = loader.load_capacities()
df_zones = loader.load_bidding_zones()

In [14]:
# ============================================================
# 3. BASIC SHAPES
# ============================================================

print("Prices:", df_prices.shape)
print("Volumes:", df_volumes.shape)
print("Flows:", df_flows.shape)
print("Capacities:", df_capacities.shape)
print("Zones:", df_zones.shape)

Prices: (1007980, 5)
Volumes: (350880, 6)
Flows: (230470, 6)
Capacities: (35132, 7)
Zones: (20, 4)


In [16]:
# ============================================================
# 4. COLUMNS AND FIRST ROWS
# ============================================================

print("Flows columns:")
print(df_flows.columns)

print("\nCapacities columns:")
print(df_capacities.columns)

print("\nFlows head:")
display(df_flows.head())

print("\nCapacities head:")
display(df_capacities.head())

Flows columns:
Index(['flow_id', 'from_zone_id', 'to_zone_id', 'delivery_day', 'hour',
       'flow_value'],
      dtype='str')

Capacities columns:
Index(['capacity_id', 'capacity_code', 'from_zone_id', 'to_zone_id',
       'delivery_day', 'hour', 'capacity_value'],
      dtype='str')

Flows head:


,flow_id,from_zone_id,to_zone_id,delivery_day,hour,flow_value
0,1,8,13,2020-01-01,0,0.0
1,2,9,13,2020-01-01,0,0.0
2,3,11,15,2020-01-01,0,0.0
3,4,12,13,2020-01-01,0,0.0
4,5,12,14,2020-01-01,0,200.0



Capacities head:


,capacity_id,capacity_code,from_zone_id,to_zone_id,delivery_day,hour,capacity_value
0,1,NO1->NO3,12,14,2020-01-01,0,200.0
1,2,NO1->SE3,12,19,2020-01-01,0,1375.0
2,3,NO3->NO1,14,12,2020-01-01,0,-200.0
3,4,SE3->NO1,19,12,2020-01-01,0,1123.0
4,5,NO1->NO3,12,14,2020-01-01,1,200.0


In [17]:
# ============================================================
# 5. CHECK CONNECTIONS IN FLOWS AND CAPACITIES
# ============================================================

flow_connections = (
    df_flows[["from_zone_id", "to_zone_id"]]
    .drop_duplicates()
    .sort_values(["from_zone_id", "to_zone_id"])
)

capacity_connections = (
    df_capacities[["from_zone_id", "to_zone_id"]]
    .drop_duplicates()
    .sort_values(["from_zone_id", "to_zone_id"])
)

print("Number of flow connections:", len(flow_connections))
print("Number of capacity connections:", len(capacity_connections))

print("\nFlow connections:")
display(flow_connections)

print("\nCapacity connections:")
display(capacity_connections)

Number of flow connections: 28
Number of capacity connections: 4

Flow connections:


,from_zone_id,to_zone_id
200902,7,13
0,8,13
1,9,13
2,11,15
3,12,13
4,12,14
5,12,16
6,12,19
200910,13,7
7,13,8



Capacity connections:


,from_zone_id,to_zone_id
0,12,14
1,12,19
2,14,12
3,19,12


In [18]:
# ============================================================
# 6. CHECK CONNECTION DIFFERENCES
# ============================================================

flow_conn_set = set(map(tuple, flow_connections.values))
capacity_conn_set = set(map(tuple, capacity_connections.values))

only_in_flows = flow_conn_set - capacity_conn_set
only_in_capacities = capacity_conn_set - flow_conn_set

print("Connections only in flows:", len(only_in_flows))
print(only_in_flows)

print("\nConnections only in capacities:", len(only_in_capacities))
print(only_in_capacities)

Connections only in flows: 24
{(np.int64(9), np.int64(13)), (np.int64(13), np.int64(7)), (np.int64(12), np.int64(16)), (np.int64(12), np.int64(13)), (np.int64(14), np.int64(16)), (np.int64(13), np.int64(16)), (np.int64(16), np.int64(12)), (np.int64(15), np.int64(18)), (np.int64(17), np.int64(15)), (np.int64(18), np.int64(15)), (np.int64(13), np.int64(8)), (np.int64(15), np.int64(11)), (np.int64(15), np.int64(14)), (np.int64(16), np.int64(13)), (np.int64(15), np.int64(17)), (np.int64(7), np.int64(13)), (np.int64(8), np.int64(13)), (np.int64(13), np.int64(9)), (np.int64(11), np.int64(15)), (np.int64(13), np.int64(12)), (np.int64(14), np.int64(18)), (np.int64(16), np.int64(14)), (np.int64(14), np.int64(15)), (np.int64(18), np.int64(14))}

Connections only in capacities: 0
set()


In [19]:
# ============================================================
# 7. CHECK ZONE ID MAPPING
# ============================================================

display(df_zones.sort_values("zone_id"))

,zone_id,zone_code,country,region_id
0,1,EE,Estonia,1
1,2,LT,Lithuania,1
2,3,LV,Latvia,1
3,4,AT,Austria,2
4,5,BE,Belgium,2
5,6,FR,France,2
6,7,GER,Germany,2
7,8,NL,Netherlands,2
8,9,DK1,Denmark,3
9,10,DK2,Denmark,3


In [20]:
# ============================================================
# 8. ADD ZONE CODES TO CONNECTIONS
# ============================================================

zone_map = df_zones.set_index("zone_id")["zone_code"].to_dict()

flow_connections_named = flow_connections.copy()
flow_connections_named["from_zone"] = flow_connections_named["from_zone_id"].map(zone_map)
flow_connections_named["to_zone"] = flow_connections_named["to_zone_id"].map(zone_map)

capacity_connections_named = capacity_connections.copy()
capacity_connections_named["from_zone"] = capacity_connections_named["from_zone_id"].map(zone_map)
capacity_connections_named["to_zone"] = capacity_connections_named["to_zone_id"].map(zone_map)

print("Flow connections with zone codes:")
display(flow_connections_named)

print("Capacity connections with zone codes:")
display(capacity_connections_named)

Flow connections with zone codes:


,from_zone_id,to_zone_id,from_zone,to_zone
200902,7,13,GER,NO2
0,8,13,NL,NO2
1,9,13,DK1,NO2
2,11,15,FI,NO4
3,12,13,NO1,NO2
4,12,14,NO1,NO3
5,12,16,NO1,NO5
6,12,19,NO1,SE3
200910,13,7,NO2,GER
7,13,8,NO2,NL


Capacity connections with zone codes:


,from_zone_id,to_zone_id,from_zone,to_zone
0,12,14,NO1,NO3
1,12,19,NO1,SE3
2,14,12,NO3,NO1
3,19,12,SE3,NO1
